In [ ]:
# =========================
# ZADATAK 1 - PSO + Neural Network (Iris Dataset)
# =========================

!pip install pyswarms tensorflow -q

import numpy as np
import pyswarms as ps

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from keras import losses

data = load_iris()

X = data.data
y = data.target

y_new = np.zeros([X.shape[0], 3])

for i in range(X.shape[0]):
    y_new[i, y[i]] = 1.

y = y_new

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

n_inputs = 4
n_hidden = 20
n_classes = 3

num_samples = 150

def probs_function(p, X):

    W1 = p[0:80].reshape((n_inputs, n_hidden))
    b1 = p[80:100].reshape((n_hidden,))

    W2 = p[100:160].reshape((n_hidden, n_classes))
    b2 = p[160:163].reshape((n_classes,))

    z1 = X.dot(W1) + b1
    a1 = np.tanh(z1)

    logits = a1.dot(W2) + b2

    exp_scores = np.exp(logits)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    return probs

def forward_prop(params):

    probs = probs_function(params, X_train)

    cce = losses.CategoricalCrossentropy()

    loss = cce(y_train, probs).numpy()

    return loss

def f(x):

    n_particles = x.shape[0]

    values = [forward_prop(x[i]) for i in range(n_particles)]

    return np.array(values)



options = {
    'c1': 0.5,
    'c2': 0.3,
    'w': 0.9
}

dimensions = (
    (n_inputs * n_hidden)
    + (n_hidden * n_classes)
    + n_hidden
    + n_classes
)

optimizer = ps.single.GlobalBestPSO(
    n_particles=100,
    dimensions=dimensions,
    options=options
)

cost, pos = optimizer.optimize(f, iters=100)

def predict(pos):

    probs = probs_function(pos, X_test)

    return probs

accuracy = (np.round(predict(pos)).astype('int') == y_test).mean()

print("Accuracy:", accuracy)